### 1.데이터 찾기
### 2.데이터 정제 및 전처리
### 3. 병합 및 집계
### 4. 인사이트 정리
- 구글 드라이브에 잘되던 안되던 제출

In [1]:
import pandas as pd


In [ ]:
waste_df = pd.read_csv("../data/전국폐기물배출업체표준데이터.csv",encoding="cp949")
waste_df.head()
waste_df.tail()
waste_df.info
waste_df.dtypes
# waste_df.shape
# waste_df.describe

<bound method NDFrame.describe of                         사업장명                    소재지도로명주소  \
0     경상남도소방본부119특수대응단119항공대   경상남도 합천군 용주면 고품부흥1길 10-28   
1                합천축협 유전자원센터    경상남도 합천군 적중면 정토양림길 42-30   
2                      대병구급대        경상남도 합천군 대병면 신성동길 23   
3                      덕곡지역대      경상남도 합천군 덕곡면 율지2길 13-9   
4                  북부119안전센터         경상남도 합천군 야로면 월광2길 5   
...                      ...                         ...   
4237                한국지역난방공사    대구광역시 달서구 달서대로 351 (대천동)   
4238                  (주)금복주      대구광역시 달서구 성서로 276 (장동)   
4239                  삼화식품공사     대구광역시 달서구 성서로 281 (갈산동)   
4240                   계명대학교  대구광역시 달서구 달구벌대로 1095 (신당동)   
4241                 (주)대한실업   대구광역시 달서구 성서로72길 82 (갈산동)   

                       소재지지번주소         위도          경도  폐기물분류명 폐기물세부종류명 처리방법내용  \
0     경상남도 합천군 용주면 고품리 910-210  35.554594  128.114688   의료폐기물      NaN    NaN   
1       경상남도 합천군 적중면 정토리 329-8  35.532190  128.260882   의료폐기물      NaN    NaN   
2 

In [ ]:
waste_df
waste_df.isna().sum() #결측치 확인 및 개수 합
# waste_df["배출량"].describe() # 배출량 기초 통계 확인
waste_df[["배출량","사업장명"]].isnull().head() ## 헤드로 확인


,배출량,사업장명
0,False,False
1,False,False
2,False,False
3,False,False
4,False,False


### 데이터 정제 
### 1. 지역별 배출량 확인
### 2. 지역별 주요 폐기물 확인
### 3. 전쳬 폐기물 분류 확인



In [ ]:
# 1개만 변수에 넘길 시 리스트 1개 사용 - 리스트 2개 사용 컬럼2개이상 시 리스트 목록으로 넘어감
# 결측치 확인
# 문자열(object) 확인
# 값이 0 또는 음수 여부 확인 
analysis_df = waste_df[["제공기관명", "폐기물분류명", "배출량"]].copy()
analysis_df["배출량"].dtype
analysis_df[["제공기관명", "폐기물분류명", "배출량"]].isna().sum() # 결측치 이상 없음


,제공기관명,폐기물분류명,배출량
0,경상남도 합천군,의료폐기물,0.360
1,경상남도 합천군,의료폐기물,1.500
2,경상남도 합천군,의료폐기물,0.360
3,경상남도 합천군,의료폐기물,0.360
4,경상남도 합천군,의료폐기물,0.360
...,...,...,...
4237,대구광역시 달서구,사업장폐기물,702091.666
4238,대구광역시 달서구,사업장폐기물,501000.000
4239,대구광역시 달서구,사업장폐기물,135000.000
4240,대구광역시 달서구,사업장폐기물,62000.000


In [ ]:
du_analysis_df=analysis_df.drop_duplicates("제공기관명") # 중복제거 후 사용, 집계 함수 사용 시 11개만 남아 있어 4천개 데이터로 나누지 못하는 상황 발생

# soft_indexs 컬럼이 아닌 라벨을 인덱스로 정렬에 사용
du_analysis_df.sort_values("배출량", ascending=False)
# sort_values 컬럼을 기준으로 정렬
sort_analysis_df=du_analysis_df.sort_values(
      by=["제공기관명", "배출량"],
      ascending=[True, False]
  )
sort_analysis_df[["제공기관명", "배출량"]].isna().sum() # 복수는 리스트 2개를 써서 컬럼 지정 후 결측치 없는 지 합계내면 된다

제공기관명    0
배출량      0
dtype: int64

In [ ]:
subset_data=sort_analysis_df.dropna(
    subset=["제공기관명","배출량"]) # dropna 함수를 사용하고 결측치 판단 파라미터 조건 전달
subset_data

,제공기관명,폐기물분류명,배출량
793,강원특별자치도 정선군,지정폐기물,2.40
2874,경기도 군포시,사업장폐기물,2500.00
1895,경기도 부천시,사업장폐기물,36.00
1702,경기도 수원시,사업장폐기물,1291.00
0,경상남도 합천군,의료폐기물,0.36
3000,대구광역시 달서구,의료폐기물,40.00
319,대구광역시 중구,의료폐기물,59.00
1364,부산광역시 금정구,사업장폐기물,150.00
1166,전북특별자치도 부안군,사업장폐기물,1.00
2569,전북특별자치도 진안군,의료폐기물,13.00


In [54]:
sort_analysis_df["배출량"].dtype

dtype('float64')

In [68]:
region_df1 = (
    analysis_df.groupby("제공기관명", as_index=False)["배출량"].sum()
)
region_df1

,제공기관명,배출량
0,강원특별자치도 정선군,1.096697e+05
1,경기도 군포시,1.579872e+07
2,경기도 부천시,1.546462e+06
3,경기도 수원시,1.748529e+07
4,경상남도 합천군,1.029182e+05
5,대구광역시 달서구,7.374182e+07
6,대구광역시 중구,7.376535e+06
7,부산광역시 금정구,7.675175e+04
8,전북특별자치도 부안군,6.254900e+04
9,전북특별자치도 진안군,3.623532e+06


In [66]:
region_df = (
    analysis_df.groupby("제공기관명", as_index=False)["배출량"].median()
)
region_df

,제공기관명,배출량
0,강원특별자치도 정선군,3.6000
1,경기도 군포시,12250.0000
2,경기도 부천시,200.0000
3,경기도 수원시,14201.0000
4,경상남도 합천군,0.5316
5,대구광역시 달서구,277.5000
6,대구광역시 중구,207.0000
7,부산광역시 금정구,20.0000
8,전북특별자치도 부안군,100.0000
9,전북특별자치도 진안군,30.0000


In [65]:
region_df = (
    analysis_df.groupby("제공기관명", as_index=False)["배출량"].mean()
)
region_df

,제공기관명,배출량
0,강원특별자치도 정선군,294.811011
1,경기도 군포시,125386.666523
2,경기도 부천시,2294.454175
3,경기도 수원시,90597.352332
4,경상남도 합천군,631.399936
5,대구광역시 달서구,59373.444172
6,대구광역시 중구,15529.547368
7,부산광역시 금정구,227.076183
8,전북특별자치도 부안군,315.904040
9,전북특별자치도 진안군,11880.432787


#지역별 배출량 합계

In [72]:
region_sum_df = (
      region_df1
      .groupby("제공기관명", as_index=False)["배출량"]
      .sum()
      .sort_values("배출량", ascending=False)
  )

region_sum_df

,제공기관명,배출량
5,대구광역시 달서구,7.374182e+07
3,경기도 수원시,1.748529e+07
1,경기도 군포시,1.579872e+07
6,대구광역시 중구,7.376535e+06
9,전북특별자치도 진안군,3.623532e+06
2,경기도 부천시,1.546462e+06
10,제주특별자치도 서귀포시,2.953944e+05
0,강원특별자치도 정선군,1.096697e+05
4,경상남도 합천군,1.029182e+05
7,부산광역시 금정구,7.675175e+04


In [ ]:
#폐기물분류명 추가하여 배출량 합계 구하기
region_category_df = (
      clean_waste_df
      .groupby(
          ["제공기관명", "폐기물분류명"],
          as_index=False
      )["배출량"]
      .sum()
  )

In [ ]:
#지역별 배출량 정렬
region_category_df = region_category_df.sort_values(
    by=["제공기관명", "배출량"],
    ascending=[True, False]
)
region_category_df

In [ ]:
#지역마다 배출량이 큰 폐기물 분류 남기기
region_top_category_df = (
    region_category_df
    .groupby("제공기관명")
    .head(1)
)

region_top_category_df

,제공기관명,폐기물분류명,배출량
0,강원특별자치도 정선군,사업장폐기물,8.714120e+04
2,경기도 군포시,사업장폐기물,1.538172e+07
3,경기도 부천시,건설폐기물,7.381868e+05
6,경기도 수원시,사업장폐기물,1.740588e+07
7,경상남도 합천군,사업장폐기물,9.985930e+04
10,대구광역시 달서구,사업장폐기물,6.670447e+07
13,대구광역시 중구,사업장폐기물,6.833375e+06
15,부산광역시 금정구,사업장폐기물,3.961442e+04
17,전북특별자치도 부안군,사업장폐기물,3.872400e+04
20,전북특별자치도 진안군,사업장폐기물,3.053697e+06


In [ ]:
### 대구 달서구가 가장 큰 배출량 1위
### 지역별로 보았을 때 사업장 폐기물이 가장 많음 
### 경기도 부천시의 경우 건설폐기물이 상당히 많은 것을 알 수 있음 

## 1. 데이터 확인

   기능          하는 일                    예시
  ━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   head()        앞부분 행 확인             waste_df.head()
  ────────────  ─────────────────────────  ───────────────────────────────
   tail()        마지막 행 확인             waste_df.tail()
  ────────────  ─────────────────────────  ───────────────────────────────
   shape         행·열 개수 확인            waste_df.shape
  ────────────  ─────────────────────────  ───────────────────────────────
   columns       컬럼 이름 확인             waste_df.columns
  ────────────  ─────────────────────────  ───────────────────────────────
   info()        자료형·결측치 요약 확인    waste_df.info()
  ────────────  ─────────────────────────  ───────────────────────────────
   describe()    숫자 열 통계 확인          waste_df["배출량"].describe()

 ## 2. 컬럼·행 선택

   기능                      하는 일                       예시
  ━━━━━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   df["컬럼"]                컬럼 하나 선택                waste_df["배출량"]
  ────────────────────────  ────────────────────────────  ────────────────────────────────────────────────────
   df[["컬럼1", "컬럼2"]]    여러 컬럼 선택                waste_df[["사업장명", "배출량"]]
  ────────────────────────  ────────────────────────────  ────────────────────────────────────────────────────
   조건 선택                 조건에 맞는 행 선택           waste_df[waste_df["배출량"] > 1000]
  ────────────────────────  ────────────────────────────  ────────────────────────────────────────────────────
   loc[]                     행 조건과 컬럼을 함께 선택    waste_df.loc[waste_df["배출량"] > 1000, ["사업장
                                                           명", "배출량"]]

  ## 3. 결측치·중복 처리

   기능                 하는 일                          예시
  ━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   isna() / isnull()    결측 여부를 True/False로 표시    waste_df.isna()
  ───────────────────  ───────────────────────────────  ──────────────────────────────
   isna().sum()         컬럼별 결측치 개수               waste_df.isna().sum()
  ───────────────────  ───────────────────────────────  ──────────────────────────────
   dropna()             결측치가 있는 행 제거            waste_df.dropna()
  ───────────────────  ───────────────────────────────  ──────────────────────────────
   fillna()             결측치를 특정 값으로 채움        waste_df["배출량"].fillna(0)
  ───────────────────  ───────────────────────────────  ──────────────────────────────
   duplicated()         중복 행 확인                     waste_df.duplicated().sum()
  ───────────────────  ───────────────────────────────  ──────────────────────────────
   drop_duplicates()    중복 행 제거                     waste_df.drop_duplicates()

  isna()와 isnull()은 같은 기능입니다. 보통 isna()를 사용하면 됩니다.

  ## 4. 정렬·변환

   기능               하는 일                             예시
  ━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   sort_values()      컬럼 값 기준 정렬                   waste_df.sort_values("배출량", ascending=False)
  ─────────────────  ──────────────────────────────────  ─────────────────────────────────────────────────────
   rename()           컬럼 이름 변경                      waste_df.rename(columns={"배출량": "배출량_톤"})
  ─────────────────  ──────────────────────────────────  ─────────────────────────────────────────────────────
   astype()           자료형 변환                         waste_df["신고기준연도"].astype(int)
  ─────────────────  ──────────────────────────────────  ─────────────────────────────────────────────────────
   pd.to_numeric()    문자열 등을 숫자로 안전하게 변환    pd.to_numeric(waste_df["배출량"], errors="coerce")
  ─────────────────  ──────────────────────────────────  ─────────────────────────────────────────────────────
   apply()            각 값에 규칙 적용                   waste_df["배출량"].apply(lambda x: "고배출" if x >=
                                                          1000 else "일반")

  ## 5. 집계·분석

   기능              하는 일                     예시
  ━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   value_counts()    값별 개수 세기              waste_df["폐기물분류명"].value_counts()
  ────────────────  ──────────────────────────  ────────────────────────────────────────────────
   groupby()         같은 기준끼리 묶기          waste_df.groupby("제공기관명")["배출량"].sum()
  ────────────────  ──────────────────────────  ────────────────────────────────────────────────
   sum()             합계                        ...["배출량"].sum()
  ────────────────  ──────────────────────────  ────────────────────────────────────────────────
   mean()            평균                        ...["배출량"].mean()
  ────────────────  ──────────────────────────  ────────────────────────────────────────────────
   median()          중앙값                      ...["배출량"].median()
  ────────────────  ──────────────────────────  ────────────────────────────────────────────────
   count()           결측치를 제외한 개수        ...["사업장명"].count()
  ────────────────  ──────────────────────────  ────────────────────────────────────────────────
   agg()             여러 집계를 한 번에 수행    groupby(...).agg({"배출량": ["sum", "mean"]})

 ## 6. 표 모양 바꾸기·합치기

   기능             하는 일                          예시
  ━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   pivot_table()    범주별 요약 표 생성              waste_df.pivot_table(index="제공기관명", columns="폐기물
                                                     분류명", values="배출량", aggfunc="sum")
  ───────────────  ───────────────────────────────  ──────────────────────────────────────────────────────────
   melt()           넓은 표를 세로형 표로 변환       table_df.melt(id_vars="제공기관명")
  ───────────────  ───────────────────────────────  ──────────────────────────────────────────────────────────
   merge()          공통 컬럼으로 두 표 연결         pd.merge(df1, df2, on="제공기관명")
  ───────────────  ───────────────────────────────  ──────────────────────────────────────────────────────────
   concat()         행 또는 열 방향으로 표 붙이기    pd.concat([df1, df2])